In [1]:
from sentence_transformers import SentenceTransformer
import chromadb
model = SentenceTransformer('all-MiniLM-L6-v2')
client_mod = chromadb.Client()
art_collection = client_mod.get_or_create_collection(name="pdf_articles_local", embedding_function=None)

c:\Users\Chris Mo\Documents\GitHub\ai4good\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def chunker(text, size=1400, repeat_len=300):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - repeat_len   
    return chunks

In [3]:
from pypdf import PdfReader

pdfread = PdfReader("../data/test.pdf")
raw_text = ""
for p in pdfread.pages:
    text = p.extract_text()
    if text:
        raw_text += text + "\n"

test_chunks = chunker(raw_text)
chunk_ids = [f"chunk-{i}" for i in range(len(test_chunks))]
embeddings = model.encode(test_chunks).tolist()

art_collection.add(ids=chunk_ids, documents=test_chunks, embeddings=embeddings)

res = art_collection.get(ids=[chunk_ids[0]], include=["embeddings"])
print(chunk_ids)
print(res["embeddings"][0][:5])


['chunk-0', 'chunk-1', 'chunk-2', 'chunk-3', 'chunk-4', 'chunk-5']
[-0.00169769 -0.03106152  0.0730658   0.07389215  0.00153441]
